# 업로드 연동 — 청크를 담을 자리 만들기어제까지는 파싱과 청킹의 결과를 화면에 찍어 보기만 했다. 셀이 끝나면 사라졌다.오늘은 그 조각들을 DB 에 남기고, 업로드 요청 하나로 파싱부터 저장까지 이어 붙인다.| 단계 | 어제 | 오늘 || --- | --- | --- || ② 파싱 | 노트북에서 호출 | 업로드 요청이 호출 || ③ 청킹 | 노트북에서 호출 | 업로드 요청이 호출 || 저장 | 없음 (메모리) | `chunks` 테이블 || 응답 | — | 작업 번호를 먼저 주고 배경에서 처리 |

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w6" / "day01"

# backend 를 import 경로에 올린다.
# .env 는 get_settings() 가 cwd 기준으로 찾으므로 ROOT 로 옮겨 온 뒤여야 한다.
# backend/ 에서 돌리면 .env 를 못 찾아 sqlite 기본값으로 떨어진다.
if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"

from app.core.config import get_settings

url = get_settings().database_url
print("프로젝트 루트 :", ROOT)
print("붙은 DB       :", url.split("@")[-1] if "@" in url else url)

프로젝트 루트 : C:\workspace\hanwha-agent
붙은 DB       : localhost:5432/agent


In [2]:
docx = next(SAMPLES.glob("DOC-FI-009_*_v1.4.docx"))
pdf = next(SAMPLES.glob("DOC-FI-009_*_v1.4.pdf"))
print("오늘 쓰는 재료")
for p in (docx, pdf):
    print(f"  {p.name:<42} {p.stat().st_size:>8,} 바이트")

오늘 쓰는 재료
  DOC-FI-009_법인카드_사용_지침_v1.4.docx              39,261 바이트
  DOC-FI-009_법인카드_사용_지침_v1.4.pdf               75,986 바이트


## 1. 담을 자리가 없다파싱하고 청킹하면 `ChunkDraft` 목록이 나온다. 변수에 담겨 있을 뿐이다.

In [3]:
from app.rag import chunker, parser

doc = parser.parse(docx)
print(f"블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}")

drafts = chunker.chunk(doc)
print(chunker.summarize(drafts))

for i, d in enumerate(drafts):
    if d.kind == "표":
        print(f"  [표] ord={i}  locator={d.locator}  글자 수={len(d.text)}")

print()
print("이 drafts 는 셀이 끝나면 사라진다. 질문이 들어올 때 다시 파싱할 수는 없다.")

C:\workspace\hanwha-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


블록 81 · 표 2 · 쪽 1
조항 단위 17 + 표 단위 2 = 19 청크
  [표] ord=1  locator=표1  글자 수=264
  [표] ord=18  locator=표2  글자 수=283

이 drafts 는 셀이 끝나면 사라진다. 질문이 들어올 때 다시 파싱할 수는 없다.


## 2. chunks 테이블`Chunk` 모델을 만들고 `models/__init__.py` 에서 import 했다.Alembic 의 autogenerate 는 `Base.metadata` 에 등록된 테이블만 본다.`__init__.py` 에 import 하지 않으면 리비전 파일에 아예 나오지 않는다.

In [4]:
import app.models

print("Base.metadata 에 등록된 표:")
for t in sorted(app.models.Base.metadata.tables):
    mark = " ←" if t == "chunks" else ""
    print(f"  {t}{mark}")

Base.metadata 에 등록된 표:
  chunks ←
  departments
  document_versions
  documents
  run_steps
  runs
  usage_logs
  users


In [5]:
# 청크는 문서가 아니라 '버전' 에 딸린다.
from app.models import Chunk

for name, col in Chunk.__table__.columns.items():
    print(f"  {name:<12} {str(col.type):<14} {'FK -> ' + list(col.foreign_keys)[0].target_fullname if col.foreign_keys else ''}")

  id           INTEGER        
  version_id   INTEGER        FK -> document_versions.id
  ord          INTEGER        
  kind         VARCHAR(16)    
  locator      VARCHAR(200)   
  text         TEXT           


## 3. 업로드 한 번으로 파싱부터 저장까지`POST /api/v1/documents` 가 파일을 받아 저장하고, 작업 번호를 돌려준다.파싱·청킹은 응답을 보낸 뒤 배경에서 돈다.재실행할 수 있게 버전은 `v9.9` 로 올리고 맨 끝 셀에서 지운다.

In [6]:
from fastapi.testclient import TestClient
from app.main import app

TEST_VER = "v9.9"
client = TestClient(app)

res = client.post(
    "/api/v1/documents",
    data={
        "doc_id": "DOC-FI-009", "title": "법인카드 사용 지침", "dept_id": "PU",
        "security_level": "일반", "version": TEST_VER, "effective_from": "2025-07-01",
    },
    files={"file": (docx.name, docx.read_bytes())},
    headers={"X-Emp-No": "2013-0177"},
)
print("상태코드:", res.status_code)
body = res.json()
for k, v in body.items():
    print(f"  {k:<12} {v}")
job_id = body["job_id"]

18:00:31 INFO     api.req.fc4992a3 문서 파일 저장: uploads\DOC-FI-009_v9.9.docx (일반)
18:00:31 INFO     httpx2 HTTP Request: POST http://testserver/api/v1/documents "HTTP/1.1 201 Created"
상태코드: 201
  doc_id       DOC-FI-009
  title        법인카드 사용 지침
  version      v9.9
  file_format  docx
  file_path    uploads/DOC-FI-009_v9.9.docx
  created      True
  job_id       b7eec29f


In [7]:
job = client.get(f"/api/v1/documents/jobs/{job_id}").json()

print(f"{job['status']} · {job['progress']}% · {job['chunk_count']}청크")
print(job["message"])
print()
for s in job["steps"]:
    print(f"  [{s['state']:>4}] {s['name']:<26} {s.get('time', '')}")
print()
print("뒤 2단계는 구현이 없어 todo 로 남는다. 그래서 완료인데 66% 다.")

18:00:31 INFO     httpx2 HTTP Request: GET http://testserver/api/v1/documents/jobs/b7eec29f "HTTP/1.1 200 OK"
완료 · 66% · 19청크
조항 단위 17 + 표 단위 2 = 19 청크

  [  ok] 파일 검증                      
  [  ok] 문서 파싱                      표 2개 인식
  [  ok] 표 -> 마크다운 변환               표 2개
  [  ok] 청킹                         19청크
  [todo] 임베딩                        
  [todo] 검색 반영 및 현행 버전 지정           

뒤 2단계는 구현이 없어 todo 로 남는다. 그래서 완료인데 66% 다.


### TestClient 는 배경 작업을 먼저 끝낸다위 셀에서 작업이 이미 `완료` 로 나왔다. 배경으로 돌린 효과가 안 보인다.`TestClient` 는 응답을 돌려주기 전에 배경 작업을 다 돌리기 때문이다.브라우저로 올리면 응답이 먼저 오고 진행률이 올라간다.

In [8]:
# 없는 작업 번호를 물으면
r = client.get("/api/v1/documents/jobs/deadbeef")
print(r.status_code, r.json())

18:00:31 INFO     httpx2 HTTP Request: GET http://testserver/api/v1/documents/jobs/deadbeef "HTTP/1.1 404 Not Found"
404 {'code': 'not_found', 'message': '작업을 찾을 수 없습니다: deadbeef', 'detail': None}


## 4. DB 에 정말 들어갔는지응답을 믿지 않고 테이블을 직접 읽는다.

In [9]:
from sqlalchemy import func, select

from app.db.session import session_scope
from app.models import DocumentVersion

with session_scope() as s:
    dv = s.scalars(
        select(DocumentVersion).where(
            DocumentVersion.doc_id == "DOC-FI-009",
            DocumentVersion.version == TEST_VER,
        )
    ).one()
    print(f"document_versions : id={dv.id} chunk_count={dv.chunk_count} "
          f"index_status={dv.index_status} progress={dv.index_progress} indexed_at={dv.indexed_at}")

    rows = s.scalars(
        select(Chunk).where(Chunk.version_id == dv.id).order_by(Chunk.ord)
    ).all()
    print(f"chunks 실제 행 수   : {len(rows)}")
    print()
    for c in rows[:4]:
        print(f"  ord={c.ord:<2} {c.kind}  {c.locator:<22} {len(c.text):>4}자  {c.text[:34]!r}")
    print("  ...")
    for c in rows:
        if c.kind == "표":
            print(f"  ord={c.ord:<2} {c.kind}  {c.locator:<22} {len(c.text):>4}자")
    print()
    longest = max(rows, key=lambda c: len(c.text))
    print(f"가장 긴 조각 {len(longest.text)}자 — locator 칼럼이 String(200) 이라서")
    print("본문은 Text 로 뒀다. String 이면 이 조각이 잘린다.")

document_versions : id=12 chunk_count=19 index_status=완료 progress=100 indexed_at=2026-10-06
chunks 실제 행 수   : 19

  ord=0  조항  DOC-FI-009_v9.9         114자  '일반\n법인카드 사용 지침\nv1.4\n문서번호  DOC-FI-00'
  ord=1  표  표1                      264자  '| 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 '
  ord=2  조항  DOC-FI-009_v9.9         205자  '이 표는 문서관리규정 제9조에 따라 관리한다.\n목차\n제1조(목'
  ord=3  조항  제1장 제1조(목적)              73자  '제1조(목적) 이 지침은 법인카드의 발급·사용·관리 및 증빙에'
  ...
  ord=1  표  표1                      264자
  ord=18 표  표2                      283자

가장 긴 조각 283자 — locator 칼럼이 String(200) 이라서
본문은 Text 로 뒀다. String 이면 이 조각이 잘린다.


## 5. 같은 판을 다시 올리면더하면 같은 조항이 두 벌 남아 검색이 같은 근거를 중복으로 집는다.그래서 `ingest_document()` 는 기존 청크를 지우고 다시 넣는다. 갈아끼운다.

In [10]:
from app.services import document_service

UPLOADED = f"uploads/DOC-FI-009_{TEST_VER}.docx"

with session_scope() as s:
    before = s.scalar(select(func.count()).select_from(Chunk))

again = document_service.ingest_document(doc_id="DOC-FI-009", version=TEST_VER, path=UPLOADED)

with session_scope() as s:
    after = s.scalar(select(func.count()).select_from(Chunk))

print("재적재 결과:", again)
print(f"chunks 전체 행 수 : {before} → {after}")
print("늘지 않았다. 더한 게 아니라 갈아끼웠다." if before == after else "늘었다 — 삭제가 안 됐다")

재적재 결과: {'chunks': 19, 'tables': 2, 'summary': '조항 단위 17 + 표 단위 2 = 19 청크'}
chunks 전체 행 수 : 19 → 19
늘지 않았다. 더한 게 아니라 갈아끼웠다.


## 6. 왜 배경으로 돌리는가지금 재료는 빠르다. 그래서 동기로 불러도 티가 안 난다.문제는 '지금 재료' 가 기준이 아니라는 점이다.

In [11]:
import time

PATTERNS = [
    "DOC-FI-009_*_v1.4.docx", "DOC-FI-009_*_v1.4.pdf",
    "DOC-HR-014_*_v2.0.docx", "DOC-HR-014_*_v2.0.pdf",
]
TIMEOUT = 10.0

def 재기(path):
    started = time.perf_counter()
    made = chunker.chunk(parser.parse(path))
    return len(made), time.perf_counter() - started


# 두 번 잰다. 첫 번째에는 파서 라이브러리를 처음 import 하는 비용이 섞여 들어간다.
print(f"{'파일':<40} {'청크':>4} {'1회차':>8} {'2회차':>8}")
for pattern in PATTERNS:
    path = next(SAMPLES.glob(pattern))
    n, first = 재기(path)
    _, second = 재기(path)
    print(f"{path.name:<40} {n:>4} {first:>7.3f}초 {second:>7.3f}초")

print()
print(f"api_client 의 TIMEOUT = {TIMEOUT}초. 재료는 전부 그 아래다.")
print("PDF 1회차가 느린 것은 파싱이 아니라 pdfplumber 를 처음 올리는 값이다.")
print("쪽수가 수백인 문서가 들어오면 요청이 그만큼 매달린다. 그때는 응답이 먼저 나가야 한다.")

파일                                         청크      1회차      2회차
DOC-FI-009_법인카드_사용_지침_v1.4.docx            19   0.012초   0.013초
DOC-FI-009_법인카드_사용_지침_v1.4.pdf             17   0.116초   0.037초
DOC-HR-014_국내출장_여비_규정_v2.0.docx            53   0.040초   0.059초
DOC-HR-014_국내출장_여비_규정_v2.0.pdf             45   0.122초   0.132초

api_client 의 TIMEOUT = 10.0초. 재료는 전부 그 아래다.
PDF 1회차가 느린 것은 파싱이 아니라 pdfplumber 를 처음 올리는 값이다.
쪽수가 수백인 문서가 들어오면 요청이 그만큼 매달린다. 그때는 응답이 먼저 나가야 한다.


## 7. 짚어 둔 것오늘 코드를 넣으면서 눈에 걸린 세 가지. 고치지 않고 남겨 둔다.

In [12]:
# (1) 제목 없는 조각의 출처는 '파일 이름' 이다.
#     업로드는 파일명을 {doc_id}_{version} 으로 갈아치운다.
#     같은 문서인데 저장되는 출처가 달라진다.
원본 = docx
업로드본 = pathlib.Path(UPLOADED)

for 이름, p in (("원본 파일명", 원본), ("업로드가 바꾼 이름", 업로드본)):
    made = chunker.chunk(parser.parse(p))
    제목없음 = {d.locator for d in made if d.kind == "조항" and "조(" not in d.locator}
    print(f"{이름:<18} {p.name:<42} → {sorted(제목없음)}")

원본 파일명             DOC-FI-009_법인카드_사용_지침_v1.4.docx            → ['DOC-FI-009_법인카드_사용_지침_v1.4']
업로드가 바꾼 이름         DOC-FI-009_v9.9.docx                       → ['DOC-FI-009_v9.9']


In [13]:
# (2) 임베딩을 안 했는데 index_status 가 '완료' 다.
#     is_searchable 은 그 값만 보므로, 벡터가 없는 판도 검색 대상으로 판정된다.
with session_scope() as s:
    dv = s.scalars(
        select(DocumentVersion).where(
            DocumentVersion.doc_id == "DOC-FI-009",
            DocumentVersion.version == TEST_VER,
        )
    ).one()
    print(f"status={dv.status} index_status={dv.index_status} → is_searchable={dv.is_searchable}")
    print("임베딩 벡터는 한 건도 없다. ④임베딩을 붙일 때 이 두 줄을 뒤로 옮겨야 한다.")

# (3) _JOBS 는 프로세스 메모리다.
print()
print("_JOBS 에 들어 있는 작업 수:", len(document_service._JOBS))
print("서버를 재시작하면 0 이 된다. --reload 가 코드 변경으로 다시 떠도 사라진다.")

status=현행 index_status=완료 → is_searchable=True
임베딩 벡터는 한 건도 없다. ④임베딩을 붙일 때 이 두 줄을 뒤로 옮겨야 한다.

_JOBS 에 들어 있는 작업 수: 1
서버를 재시작하면 0 이 된다. --reload 가 코드 변경으로 다시 떠도 사라진다.


## 정리테스트로 만든 `v9.9` 를 지운다. `cascade="all, delete-orphan"` 이 걸려 있어버전을 지우면 청크도 함께 지워진다.

In [14]:
from app.models import Document

with session_scope() as s:
    dv = s.scalars(
        select(DocumentVersion).where(
            DocumentVersion.doc_id == "DOC-FI-009",
            DocumentVersion.version == TEST_VER,
        )
    ).one_or_none()
    if dv is not None:
        s.delete(dv)
with session_scope() as s:
    doc = s.get(Document, "DOC-FI-009")
    if doc is not None and not doc.versions:
        s.delete(doc)
    남은 = s.scalar(select(func.count()).select_from(Chunk))

pathlib.Path(UPLOADED).unlink(missing_ok=True)
print("chunks 남은 행 수:", 남은, "— 버전을 지우니 청크도 함께 지워졌다")
print("업로드 파일도 삭제했다. 이 노트북은 처음부터 다시 돌릴 수 있다.")

chunks 남은 행 수: 0 — 버전을 지우니 청크도 함께 지워졌다
업로드 파일도 삭제했다. 이 노트북은 처음부터 다시 돌릴 수 있다.
